# Form Analyzer (TF pose sequences)

# Form Analyzer — TF temporal pose model (ST-GCN / 1D-CNN over 17-joint sequences)

Processes `training/process_form_data.py` output (`data/processed/forms/normalised.json`):
normalised 17-joint keypoint sequences + per-exercise quality labels
(`good` / `slight_misalignment` / `bad`). Matches the serving contract of
`app/form_analyzer_engine.py` (shoulder-width = 1, origin = hip midpoint).

In [1]:
import importlib.util
import os
import pathlib
import sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
sys.path.insert(0, str(ai))         # for the training.* namespace package
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
# -- shared bootstrap: seeds RNGs, caps CPU threads, resolves data/output roots ----
# (must run before TensorFlow is imported so thread env applies)
try:
    from training.bootstrap import *  # noqa: F403
    CFG = init()
except Exception as _boot_err:  # bootstrap is an upgrade, never brick a notebook
    print('[bootstrap] unavailable:', repr(_boot_err))
    CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))   # smoke | demo | full

from tf_utils import set_memory_growth
set_memory_growth()



2026-08-04 23:04:31.492863: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-04 23:04:31.592876: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-04 23:04:33.616133: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-04 23:04:36.233119: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Load pose-keypoint sequences (extract from workout videos if missing/stale)
import os
import sys
import json
import subprocess
from pathlib import Path
from collections import Counter

proc = Path('../data/processed/forms/normalised.json')
if not proc.exists() or os.environ.get('BUDDY_REPROCESS') == '1':
    print('Extracting pose keypoints from the workout-video datasets...')
    subprocess.run([sys.executable, '../training/process_workout_videos.py',
                    '--frames', '24'], check=True)

rows = json.loads(proc.read_text())
print('sequences:', len(rows))
classes = sorted({r['exercise'] for r in rows})
print('classes:', len(classes))
print('distribution:', dict(Counter(r['exercise'] for r in rows)))

sequences: 22
classes: 22
distribution: {'barbell biceps curl': 1, 'bench press': 1, 'chest fly machine': 1, 'deadlift': 1, 'decline bench press': 1, 'hammer curl': 1, 'hip thrust': 1, 'incline bench press': 1, 'lat pulldown': 1, 'lateral raise': 1, 'leg extension': 1, 'leg raises': 1, 'plank': 1, 'pull Up': 1, 'push-up': 1, 'romanian deadlift': 1, 'russian twist': 1, 'shoulder press': 1, 'squat': 1, 't bar row': 1, 'tricep Pushdown': 1, 'tricep dips': 1}


In [3]:
# Stratified train/val/test split + fixed-length padding (T, 17, 3)
import numpy as np
from sklearn.model_selection import train_test_split

T = {'smoke': 16, 'demo': 32, 'full': 48}[SCALE]
def pad(seq):
    out = np.zeros((T, 17, 3), dtype=np.float32)
    s = np.asarray(seq[:T], dtype=np.float32)
    out[:len(s)] = s
    return out

X = np.stack([pad(r['normalised']) for r in rows])
y = np.array([classes.index(r['exercise']) for r in rows], dtype=np.int32)

idx = np.arange(len(rows))
# Fallback to random split when any class has < 2 members (smoke uses 1/class)
from collections import Counter
min_per_class = min(Counter(r['exercise'] for r in rows).values())
strat = y if min_per_class >= 2 else None

train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=strat)
val_idx, test_idx = train_test_split(val_idx, test_size=0.5, random_state=42,
                                     stratify=strat[val_idx] if strat is not None else None)
print('train/val/test:', len(train_idx), len(val_idx), len(test_idx))

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y[train_idx]), y=y[train_idx])
class_weight = dict(enumerate(cw))

train/val/test: 17 2 3


In [4]:
# Temporal model: per-joint Conv head -> BiLSTM over the sequence
import tensorflow as tf
from tf_utils import set_memory_growth
set_memory_growth()

inp = tf.keras.Input(shape=(T, 17, 3))
x = tf.keras.layers.Conv2D(32, (1, 3), padding='same', activation='relu')(inp)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Conv2D(64, (1, 1), padding='same', activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Reshape((T, 17 * 64))(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(len(classes), activation='softmax')(x)
m = tf.keras.Model(inp, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy',
          metrics=['accuracy'])
m.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 17, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 17, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 17, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 17, 64)     │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 17, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 32, 1088)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32, 128)        │       590,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 22)             │         1,430 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 602,838 (2.30 MB)

 Trainable params: 602,646 (2.30 MB)

 Non-trainable params: 192 (768.00 B)

In [8]:
# Train (early-stopped) with class weights + LR decay
import time
EPOCHS = {'smoke': 2, 'demo': 300, 'full': 60}[SCALE]
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=2, min_lr=1e-5),
    tf.keras.callbacks.TensorBoard(log_dir=f'../logs/form_analyzer-{SCALE}'),
]
t0 = time.time()
m.fit(X[train_idx], y[train_idx], epochs=EPOCHS, batch_size=16,
      validation_data=(X[val_idx], y[val_idx]),
      class_weight=class_weight, callbacks=callbacks, verbose=1)
print(f'total {time.time()-t0:.0f}s')

Epoch 1/300


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 273ms/step - accuracy: 0.1176 - loss: 3.0120 - val_accuracy: 0.0000e+00 - val_loss: 3.1457 - learning_rate: 2.5000e-04
Epoch 2/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.1176 - loss: 2.8337 - val_accuracy: 0.0000e+00 - val_loss: 3.1461 - learning_rate: 2.5000e-04
Epoch 3/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - accuracy: 0.1176 - loss: 2.7344 - val_accuracy: 0.0000e+00 - val_loss: 3.1444 - learning_rate: 2.5000e-04
Epoch 4/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.1765 - loss: 2.8616 - val_accuracy: 0.0000e+00 - val_loss: 3.1461 - learning_rate: 2.5000e-04
Epoch 5/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.1176 - loss: 2.9257 - val_accuracy: 0.0000e+00 - val_loss: 3.1532 - learning_rate: 2.5000e-04
Epoch 6/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - accuracy: 0.1765 - loss: 2.7297 - val_accuracy: 0.0000e+00 - val_loss: 3.1600 - learning_rate: 1.2500e-04
Epoch 7/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accur

In [6]:
# Evaluate on the untouched test split + overfit gauge
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

p_test = m.predict(X[test_idx], verbose=0).argmax(1)
acc = float(accuracy_score(y[test_idx], p_test))
bal = float(balanced_accuracy_score(y[test_idx], p_test))
print(f'test n={len(test_idx)} acc={acc:.3f} balanced_acc={bal:.3f}')
print('confusion (rows=truth, cols=pred):')
print(confusion_matrix(y[test_idx], p_test))

p_train = m.predict(X[train_idx], verbose=0).argmax(1)
train_acc = float(accuracy_score(y[train_idx], p_train))
print(f'overfit gauge: train acc={train_acc:.3f} | test acc={acc:.3f} | gap={train_acc - acc:+.3f}')

/home/peter/Desktop/ml-env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


test n=3 acc=0.000 balanced_acc=0.000
confusion (rows=truth, cols=pred):
[[0 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 1]
 [1 0 0 0 0]
 [0 0 0 0 0]]
overfit gauge: train acc=0.176 | test acc=0.000 | gap=+0.176


In [7]:
# Export ONNX (+ INT8) + persist the joint spec + log the run
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(m, Path('../models'), 'form_analyzer', '1.0.0',
    input_signature=[tf.TensorSpec((None, T, 17, 3), tf.float32, name='pose_input')])
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'form_analyzer', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'scenario': 'workout_videos', 'n_classes': len(classes),
            'metrics': {'test_accuracy': acc, 'balanced_accuracy': bal,
                        'train_accuracy': train_acc}})
print('exported', q)

I0000 00:00:1785873888.821697 1055206 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785873888.821855 1055206 single_machine.cc:376] Starting new session
I0000 00:00:1785873889.123701 1055206 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785873889.124085 1055206 single_machine.cc:376] Starting new session
I0000 00:00:1785873889.419264 1055206 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


{
  "name": "form_analyzer",
  "version": "1.0.0",
  "artifact_path": "../models/form_analyzer-1.0.0_int8.onnx",
  "framework": "tensorflow",
  "scenario": "workout_videos",
  "n_classes": 22,
  "metrics": {
    "test_accuracy": 0.0,
    "balanced_accuracy": 0.0,
    "train_accuracy": 0.17647058823529413
  }
}
exported ../models/form_analyzer-1.0.0_int8.onnx
